# HW14: эмбеддинги, FAISS, оценка retrieval, обновление базы и mini-RAG

Учебный pipeline: база знаний → чанки → эмбеддинги (`sentence-transformers`) → индекс FAISS → метрики → эксперимент по `chunk_size` → обновление корпуса → mini-RAG. Артефакты: `./artifacts/`. Запускайте **Run All** из каталога `homeworks/HW14` (текущая папка = `HW14`).


## Импорты, seed и среда


In [1]:
import random
import re
import subprocess
import sys
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

try:
    import faiss
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"])
    import faiss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display

try:
    import torch
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
    import torch

try:
    from sentence_transformers import SentenceTransformer
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
    from sentence_transformers import SentenceTransformer

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["font.size"] = 10


In [2]:
SEED = 42


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__, "| device:", DEVICE)
print("sklearn:", sklearn.__version__)

ROOT = Path.cwd()
if not (ROOT / "artifacts").exists() and (ROOT / "homeworks" / "HW14" / "artifacts").exists():
    ROOT = ROOT / "homeworks" / "HW14"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT.resolve(), "| ARTIFACTS:", ARTIFACTS.resolve())


torch: 2.10.0+cpu | device: cpu
sklearn: 1.6.1
ROOT: C:\Users\Гошанский\PycharmProjects\mirea-aie\homeworks\HW14 | ARTIFACTS: C:\Users\Гошанский\PycharmProjects\mirea-aie\homeworks\HW14\artifacts


## База знаний и первичный анализ

Тематика: **DevOps, окружения и воспроизводимость** учебных ML-проектов (Git, venv/conda, Docker, CI, тесты, секреты, логи). Документы связаны одной областью; по ним естественны вопросы и извлечение фрагментов для ответа — разумный учебный retrieval / mini-RAG. Корпус задан в коде (без внешних файлов), загрузка полностью воспроизводима.


In [3]:
RAW_DOCS: List[Dict[str, str]] = [
    {
        "doc_id": "git_basics",
        "title": "Git: коммиты и ветки",
        "text": (
            "Git хранит снимки состояния файлов и историю изменений. Коммит фиксирует набор правок с сообщением. "
            "Ветка — это указатель на коммит; слияние веток объединяет истории. Для учебных проектов полезно делать частые "
            "небольшие коммиты и осмысленные сообщения. Команда git status показывает рабочее дерево, git diff — несохранённые изменения."
        ),
    },
    {
        "doc_id": "git_remote",
        "title": "Git: удалённые репозитории",
        "text": (
            "Удалённый репозиторий — копия проекта на сервере. git remote add задаёт псевдоним origin. "
            "Команды git fetch и git pull подтягивают изменения; git push отправляет локальные коммиты. При конфликте слияния "
            "нужно вручную разрешить помеченные участки в файлах и завершить merge коммитом."
        ),
    },
    {
        "doc_id": "venv",
        "title": "Виртуальное окружение Python (venv)",
        "text": (
            "Модуль venv создаёт изолированный интерпретатор и каталог site-packages. Активация на Windows: "
            ".\\venv\\Scripts\\activate, на Unix: source venv/bin/activate. После активации pip устанавливает пакеты только в окружение. "
            "Файл requirements.txt фиксирует версии зависимостей для воспроизводимости."
        ),
    },
    {
        "doc_id": "conda",
        "title": "Conda-окружения",
        "text": (
            "Conda управляет пакетами и окружениями, включая бинарные зависимости вне pip. conda create -n myenv python=3.11 "
            "создаёт окружение; conda activate myenv включает его. environment.yml описывает окружение для коллаборации. "
            "Для ML-проектов conda удобна, когда нужны CUDA-библиотеки и несовместимые версии системных пакетов."
        ),
    },
    {
        "doc_id": "docker_intro",
        "title": "Docker: образ и контейнер",
        "text": (
            "Образ Docker — неизменяемый шаблон файловой системы и метаданных. Контейнер — запущенный экземпляр образа "
            "с изолированным процессом. Dockerfile описывает слои: FROM, RUN, COPY, CMD. Слои кешируются при пересборке. "
            "Контейнеры легче виртуальных машин, так как делят ядро ОС хоста."
        ),
    },
    {
        "doc_id": "docker_compose",
        "title": "Docker Compose",
        "text": (
            "Compose задаёт multi-container приложение в YAML: сервисы, сети, тома. docker compose up поднимает стек, "
            "docker compose down останавливает. Переменные окружения задаются в .env или секции environment. Для локальной разработки "
            "удобно монтировать исходники как volume для hot-reload без пересборки образа."
        ),
    },
    {
        "doc_id": "ci_cd",
        "title": "CI/CD: непрерывная интеграция",
        "text": (
            "CI автоматически собирает и тестирует код при каждом push. Типичный pipeline: checkout, установка зависимостей, "
            "линтер, unit-тесты, артефакты. CD расширяет процесс выкладкой в staging или production. Матричные сборки проверяют код "
            "на нескольких версиях Python или ОС."
        ),
    },
    {
        "doc_id": "github_actions",
        "title": "GitHub Actions",
        "text": (
            "Workflow — YAML в .github/workflows. Триггеры: push, pull_request, schedule. job содержит steps; "
            "действия — готовые шаги из marketplace. Секреты хранятся в настройках репозитория и подставляются как ${{ secrets.NAME }}. "
            "Кеширование зависимостей ускоряет повторные запуски."
        ),
    },
    {
        "doc_id": "pytest",
        "title": "Тестирование с pytest",
        "text": (
            "pytest собирает тесты из файлов test_*.py и функций test_*. Фикстуры переиспользуют подготовку данных. "
            "Параметризация @pytest.mark.parametrize прогоняет один тест с разными входами. Покрытие кода измеряют pytest-cov; "
            "порог покрытия можно включить в CI как gate."
        ),
    },
    {
        "doc_id": "precommit",
        "title": "Pre-commit хуки",
        "text": (
            "pre-commit запускает проверки перед коммитом: форматирование, линтер, секрет-сканеры. Конфиг .pre-commit-config.yaml "
            "перечисляет репозитории и хуки. Установка: pre-commit install. Это снижает шум в code review и ловит ошибки локально."
        ),
    },
    {
        "doc_id": "makefile",
        "title": "Makefile для задач",
        "text": (
            "Makefile задаёт цели и рецепты; make train вызывает нужную команду без запоминания длинных строк. "
            "Переменные и phony-цели делают сценарии переносимыми между ОС при наличии make. Для кроссплатформенности иногда используют "
            "invoke или just вместо make."
        ),
    },
    {
        "doc_id": "secrets",
        "title": "Секреты и .env",
        "text": (
            "Секретные ключи не коммитят в Git. Локально их держат в .env, который в .gitignore. В приложении загрузка через "
            "python-dotenv или переменные окружения в контейнере. В продакшене — менеджеры секретов Vault или облачные Secret Manager."
        ),
    },
    {
        "doc_id": "logging",
        "title": "Логирование",
        "text": (
            "Стандартный модуль logging даёт уровни DEBUG, INFO, WARNING, ERROR. Настройка handlers: консоль, файл, ротация. "
            "Структурированные JSON-логи упрощают поиск в системах вроде ELK. В сервисах важно логировать correlation id запроса."
        ),
    },
    {
        "doc_id": "reproducibility",
        "title": "Воспроизводимость экспериментов",
        "text": (
            "Для ML фиксируют seed, версии библиотек, hash датасета и конфиг обучения. Docker и lock-файлы зависимостей "
            "уменьшают дрейф окружения. Сохранение чекпоинта и метрик рядом с git-тегом версии модели облегчает аудит."
        ),
    },
    {
        "doc_id": "gpu_docker",
        "title": "GPU в Docker",
        "text": (
            "Для NVIDIA используют nvidia-container-toolkit; в compose задаётся deploy.resources.reservations.devices. "
            "Образы с CUDA должны соответствовать версии драйвера хоста. Для обучения проверяют nvidia-smi внутри контейнера."
        ),
    },
]

docs_df = pd.DataFrame(RAW_DOCS)
print("Число документов:", len(RAW_DOCS))
display(docs_df[["doc_id", "title"]])
for i, row in enumerate(RAW_DOCS[:4]):
    print(f"\n--- Пример {i+1}: {row['doc_id']} ---\n{row['text'][:420]}...")


Число документов: 15


,doc_id,title
0,git_basics,Git: коммиты и ветки
1,git_remote,Git: удалённые репозитории
2,venv,Виртуальное окружение Python (venv)
3,conda,Conda-окружения
4,docker_intro,Docker: образ и контейнер
5,docker_compose,Docker Compose
6,ci_cd,CI/CD: непрерывная интеграция
7,github_actions,GitHub Actions
8,pytest,Тестирование с pytest
9,precommit,Pre-commit хуки



--- Пример 1: git_basics ---
Git хранит снимки состояния файлов и историю изменений. Коммит фиксирует набор правок с сообщением. Ветка — это указатель на коммит; слияние веток объединяет истории. Для учебных проектов полезно делать частые небольшие коммиты и осмысленные сообщения. Команда git status показывает рабочее дерево, git diff — несохранённые изменения....

--- Пример 2: git_remote ---
Удалённый репозиторий — копия проекта на сервере. git remote add задаёт псевдоним origin. Команды git fetch и git pull подтягивают изменения; git push отправляет локальные коммиты. При конфликте слияния нужно вручную разрешить помеченные участки в файлах и завершить merge коммитом....

--- Пример 3: venv ---
Модуль venv создаёт изолированный интерпретатор и каталог site-packages. Активация на Windows: .\venv\Scripts\activate, на Unix: source venv/bin/activate. После активации pip устанавливает пакеты только в окружение. Файл requirements.txt фиксирует версии зависимостей для воспроизводимости...

## Чанкинг

Скользящее окно по **символам**: `CHUNK_SIZE`, перекрытие `CHUNK_OVERLAP`. Короткий документ даёт один чанк. Параметры выбраны так, чтобы получить порядка **30–80** фрагментов и не резать слишком агрессивно.


In [4]:
CHUNK_SIZE = 380
CHUNK_OVERLAP = 90


def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    text = re.sub(r"\s+", " ", text.strip())
    if len(text) <= chunk_size:
        return [text]
    chunks: List[str] = []
    start = 0
    step = max(1, chunk_size - overlap)
    while start < len(text):
        chunks.append(text[start : start + chunk_size])
        start += step
    return chunks


def documents_to_chunks(
    docs: Sequence[Dict[str, str]], chunk_size: int, overlap: int
) -> Tuple[List[Dict[str, object]], pd.DataFrame]:
    rows: List[Dict[str, object]] = []
    for d in docs:
        parts = chunk_text(d["text"], chunk_size=chunk_size, overlap=overlap)
        for j, ch in enumerate(parts):
            rows.append(
                {
                    "doc_id": d["doc_id"],
                    "title": d["title"],
                    "chunk_id": f"{d['doc_id']}#{j}",
                    "chunk_index": j,
                    "text": ch,
                }
            )
    chunk_df = pd.DataFrame(rows)
    return rows, chunk_df


chunks_meta, chunks_df = documents_to_chunks(RAW_DOCS, CHUNK_SIZE, CHUNK_OVERLAP)
print("Число чанков:", len(chunks_df))
display(chunks_df.head(8))

example_doc_id = "docker_intro"
ex_chunks = chunks_df[chunks_df["doc_id"] == example_doc_id]
print(f"\nДокумент '{example_doc_id}' → {len(ex_chunks)} чанков:")
for _, r in ex_chunks.iterrows():
    print(f"  [{r['chunk_id']}]: {r['text'][:200]}...")


Число чанков: 15


,doc_id,title,chunk_id,chunk_index,text
0,git_basics,Git: коммиты и ветки,git_basics#0,0,Git хранит снимки состояния файлов и историю и...
1,git_remote,Git: удалённые репозитории,git_remote#0,0,Удалённый репозиторий — копия проекта на серве...
2,venv,Виртуальное окружение Python (venv),venv#0,0,Модуль venv создаёт изолированный интерпретато...
3,conda,Conda-окружения,conda#0,0,"Conda управляет пакетами и окружениями, включа..."
4,docker_intro,Docker: образ и контейнер,docker_intro#0,0,Образ Docker — неизменяемый шаблон файловой си...
5,docker_compose,Docker Compose,docker_compose#0,0,Compose задаёт multi-container приложение в YA...
6,ci_cd,CI/CD: непрерывная интеграция,ci_cd#0,0,CI автоматически собирает и тестирует код при ...
7,github_actions,GitHub Actions,github_actions#0,0,Workflow — YAML в .github/workflows. Триггеры:...



Документ 'docker_intro' → 1 чанков:
  [docker_intro#0]: Образ Docker — неизменяемый шаблон файловой системы и метаданных. Контейнер — запущенный экземпляр образа с изолированным процессом. Dockerfile описывает слои: FROM, RUN, COPY, CMD. Слои кешируются пр...


## Эмбеддинги и индекс FAISS

Модель **paraphrase-multilingual-MiniLM-L12-v2** (рус/англ). Векторы **нормируются**; **IndexFlatIP** и скалярное произведение эквивалентны косинусу для единичных векторов.


In [5]:
EMBED_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedder = SentenceTransformer(EMBED_MODEL_NAME, device=str(DEVICE))


def encode_texts(texts: Sequence[str]) -> np.ndarray:
    emb = embedder.encode(
        list(texts),
        batch_size=32,
        convert_to_numpy=True,
        show_progress_bar=False,
        normalize_embeddings=True,
    )
    return emb.astype(np.float32)


def build_faiss_ip_index(vectors: np.ndarray) -> faiss.Index:
    dim = vectors.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(vectors)
    return index


def search_topk(
    query_emb: np.ndarray,
    index: faiss.Index,
    chunk_table: pd.DataFrame,
    k: int,
) -> pd.DataFrame:
    scores, idxs = index.search(query_emb.astype(np.float32), k)
    out = chunk_table.iloc[idxs[0]].copy()
    out["score"] = scores[0]
    return out.reset_index(drop=True)


chunk_texts = chunks_df["text"].tolist()
chunk_vectors = encode_texts(chunk_texts)
faiss_index = build_faiss_ip_index(chunk_vectors)

demo_queries = [
    "Как активировать виртуальное окружение на Windows?",
    "Что такое Docker Compose и зачем volume?",
    "Как в GitHub Actions хранить секреты?",
    "Зачем нужен pre-commit?",
]

print("Примеры retrieval (top-3):\n")
for q in demo_queries:
    qv = encode_texts([q])
    hits = search_topk(qv, faiss_index, chunks_df, k=3)
    print("Q:", q)
    for _, h in hits.iterrows():
        print(f"  {h['doc_id']} | score={h['score']:.3f} | {h['text'][:120]}...")
    print()


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Примеры retrieval (top-3):

Q: Как активировать виртуальное окружение на Windows?
  venv | score=0.434 | Модуль venv создаёт изолированный интерпретатор и каталог site-packages. Активация на Windows: .\venv\Scripts\activate, ...
  conda | score=0.383 | Conda управляет пакетами и окружениями, включая бинарные зависимости вне pip. conda create -n myenv python=3.11 создаёт ...
  docker_compose | score=0.332 | Compose задаёт multi-container приложение в YAML: сервисы, сети, тома. docker compose up поднимает стек, docker compose ...

Q: Что такое Docker Compose и зачем volume?
  docker_intro | score=0.657 | Образ Docker — неизменяемый шаблон файловой системы и метаданных. Контейнер — запущенный экземпляр образа с изолированны...
  docker_compose | score=0.450 | Compose задаёт multi-container приложение в YAML: сервисы, сети, тома. docker compose up поднимает стек, docker compose ...
  reproducibility | score=0.267 | Для ML фиксируют seed, версии библиотек, hash датасета и конфиг обучения. D

## Контрольные запросы и оценка retrieval

**10** запросов; для каждого задан золотой `doc_id`. **hit@k** — есть ли золотой документ среди уникальных `doc_id` в top-k; **recall@k** — доля чанков золотого документа среди всех его чанков, попавших в top-k (не выше 1). Дополнительно **MRR@k** и **rank_of_first_relevant**.


In [6]:
K_EVAL = 5

CONTROL_QUERIES: List[Dict[str, str]] = [
    {"query": "Как добавить удалённый репозиторий и отправить коммиты?", "expected": "git_remote"},
    {"query": "Где хранить API-ключи и как подключить dotenv?", "expected": "secrets"},
    {"query": "Как создать conda-окружение с Python 3.11?", "expected": "conda"},
    {"query": "Что такое слой в Dockerfile и как кешируется сборка?", "expected": "docker_intro"},
    {"query": "Как поднять несколько сервисов одной командой локально?", "expected": "docker_compose"},
    {"query": "Что запускается при git push в CI?", "expected": "ci_cd"},
    {"query": "Как параметризовать один тест в pytest?", "expected": "pytest"},
    {"query": "Зачем фиксировать seed и версии пакетов в ML?", "expected": "reproducibility"},
    {"query": "Как в контейнере использовать NVIDIA GPU?", "expected": "gpu_docker"},
    {"query": "Какие уровни есть у модуля logging?", "expected": "logging"},
]


def doc_chunk_counts(chunk_table: pd.DataFrame) -> Dict[str, int]:
    return chunk_table.groupby("doc_id").size().to_dict()


def evaluate_retrieval(
    queries: Sequence[Dict[str, str]],
    chunk_table: pd.DataFrame,
    index: faiss.Index,
    k: int,
) -> Tuple[pd.DataFrame, Dict[str, float]]:
    counts = doc_chunk_counts(chunk_table)
    rows_out = []
    reciprocal_ranks = []

    for item in queries:
        q = item["query"]
        gold = item["expected"]
        qv = encode_texts([q])
        hits = search_topk(qv, index, chunk_table, k=k)
        retrieved_sources = []
        seen = set()
        for _, h in hits.iterrows():
            did = h["doc_id"]
            if did not in seen:
                seen.add(did)
                retrieved_sources.append(did)

        hit_at_k = int(gold in set(hits["doc_id"].tolist()))
        gold_chunks_in_topk = int((hits["doc_id"] == gold).sum())
        denom = max(1, counts[gold])
        recall_at_k = min(1.0, gold_chunks_in_topk / denom)

        rel_positions = np.where(hits["doc_id"].values == gold)[0]
        rank_first = int(rel_positions[0] + 1) if len(rel_positions) else None
        if rank_first is not None:
            reciprocal_ranks.append(1.0 / rank_first)
        else:
            reciprocal_ranks.append(0.0)

        rows_out.append(
            {
                "query": q,
                "expected_source": gold,
                "retrieved_sources": "|".join(retrieved_sources),
                "hit_at_k": hit_at_k,
                "recall_at_k": recall_at_k,
                "rank_of_first_relevant": rank_first if rank_first is not None else "",
            }
        )

    ev_df = pd.DataFrame(rows_out)
    metrics = {
        "hit@k": ev_df["hit_at_k"].mean(),
        "recall@k": ev_df["recall_at_k"].mean(),
        f"MRR@{k}": float(np.mean(reciprocal_ranks)),
    }
    return ev_df, metrics


eval_df, metrics_main = evaluate_retrieval(CONTROL_QUERIES, chunks_df, faiss_index, K_EVAL)
print("Метрики (основная конфигурация):", metrics_main)
display(eval_df)

eval_path = ARTIFACTS / "retrieval_eval.csv"
eval_df.to_csv(eval_path, index=False, encoding="utf-8")
print("Сохранено:", eval_path)


Метрики (основная конфигурация): {'hit@k': np.float64(1.0), 'recall@k': np.float64(1.0), 'MRR@5': 0.9333333333333332}


,query,expected_source,retrieved_sources,hit_at_k,recall_at_k,rank_of_first_relevant
0,Как добавить удалённый репозиторий и отправить...,git_remote,git_remote|docker_compose|git_basics|docker_in...,1,1.0,1
1,Где хранить API-ключи и как подключить dotenv?,secrets,secrets|docker_compose|venv|conda|gpu_docker,1,1.0,1
2,Как создать conda-окружение с Python 3.11?,conda,conda|ci_cd|pytest|gpu_docker|docker_intro,1,1.0,1
3,Что такое слой в Dockerfile и как кешируется с...,docker_intro,docker_intro|docker_compose|reproducibility|ci...,1,1.0,1
4,Как поднять несколько сервисов одной командой ...,docker_compose,docker_compose|github_actions|git_remote|loggi...,1,1.0,1
5,Что запускается при git push в CI?,ci_cd,git_basics|git_remote|ci_cd|secrets|github_act...,1,1.0,3
6,Как параметризовать один тест в pytest?,pytest,pytest|ci_cd|precommit|makefile|reproducibility,1,1.0,1
7,Зачем фиксировать seed и версии пакетов в ML?,reproducibility,reproducibility|conda|pytest|ci_cd|docker_compose,1,1.0,1
8,Как в контейнере использовать NVIDIA GPU?,gpu_docker,gpu_docker|docker_intro|docker_compose|conda|s...,1,1.0,1
9,Какие уровни есть у модуля logging?,logging,logging|docker_compose|reproducibility|git_bas...,1,1.0,1


Сохранено: C:\Users\Гошанский\PycharmProjects\mirea-aie\homeworks\HW14\artifacts\retrieval_eval.csv


## Эксперимент с параметрами retrieval

Сравниваются два значения **`CHUNK_SIZE`**: 280 и 480 при фиксированном `CHUNK_OVERLAP=90`. Остальное без изменений; те же запросы и метрики.


In [7]:
def pipeline_for_chunk_size(cs: int, overlap: int = CHUNK_OVERLAP):
    _, cdf = documents_to_chunks(RAW_DOCS, cs, overlap)
    vecs = encode_texts(cdf["text"].tolist())
    idx = build_faiss_ip_index(vecs)
    return cdf, vecs, idx


exp_sizes = [280, 480]
exp_results = {}
for cs in exp_sizes:
    cdf_e, vecs_e, idx_e = pipeline_for_chunk_size(cs)
    _, m = evaluate_retrieval(CONTROL_QUERIES, cdf_e, idx_e, K_EVAL)
    exp_results[cs] = m
    print(f"chunk_size={cs} ->", m)

best_cs = max(exp_results, key=lambda x: exp_results[x]["hit@k"])
print("\nПо hit@k лучше chunk_size =", best_cs)


chunk_size=280 -> {'hit@k': np.float64(1.0), 'recall@k': np.float64(1.0), 'MRR@5': 0.875}
chunk_size=480 -> {'hit@k': np.float64(1.0), 'recall@k': np.float64(1.0), 'MRR@5': 0.9333333333333332}

По hit@k лучше chunk_size = 280


## Обновление базы знаний и переиндексация

Добавлены **три** документа (Kubernetes, Helm, Prometheus). Повторные чанкинг и построение индекса. Ниже — сравнение retrieval до/после на выбранных запросах; результаты пишутся в `artifacts/retrieval_before_after_update.csv`.


In [8]:
NEW_DOCS: List[Dict[str, str]] = [
    {
        "doc_id": "k8s_basics",
        "title": "Kubernetes: поды и деплойменты",
        "text": (
            "Kubernetes оркестрирует контейнеры на кластере. Pod — минимальная единица: один или несколько контейнеров с общим сетевым namespace. "
            "Deployment управляет репликами и rolling update. kubectl apply применяет манифесты YAML. Пробы liveness и readiness помогают перезапускать "
            "нездоровые поды и исключать их из балансировки до готовности."
        ),
    },
    {
        "doc_id": "helm",
        "title": "Helm-чарты",
        "text": (
            "Helm пакует Kubernetes-манифесты в chart с параметрами values.yaml. Команда helm install разворачивает релиз, "
            "helm upgrade обновляет его. Шаблонизация уменьшает дублирование YAML между средами staging и production."
        ),
    },
    {
        "doc_id": "prometheus",
        "title": "Prometheus и метрики",
        "text": (
            "Prometheus собирает метрики pull-моделью с /metrics endpoints. PromQL запрашивает временные ряды; Alertmanager маршрутизирует алерты. "
            "Для сервисов полезны histogram и summary для латентности. Grafana визуализирует дашборды поверх Prometheus."
        ),
    },
]

DOCS_BEFORE = list(RAW_DOCS)
DOCS_AFTER = DOCS_BEFORE + NEW_DOCS

chunks_before, chunks_df_before = documents_to_chunks(DOCS_BEFORE, CHUNK_SIZE, CHUNK_OVERLAP)
chunks_after, chunks_df_after = documents_to_chunks(DOCS_AFTER, CHUNK_SIZE, CHUNK_OVERLAP)

vec_b = encode_texts(chunks_df_before["text"].tolist())
idx_b = build_faiss_ip_index(vec_b)

vec_a = encode_texts(chunks_df_after["text"].tolist())
idx_a = build_faiss_ip_index(vec_a)

COMPARE_QUERIES = [
    "Как rolling update настроить в Kubernetes?",
    "Чем Helm отличается от простого kubectl apply?",
    "Где взять метрики латентности для алертов?",
    "Как в Docker Compose поднять стек локально?",
    "Зачем нужен pytest-cov в CI?",
]

before_after_rows = []
for q in COMPARE_QUERIES:
    qv = encode_texts([q])
    top_b = search_topk(qv, idx_b, chunks_df_before, k=5)
    top_a = search_topk(qv, idx_a, chunks_df_after, k=5)
    src_b = []
    seen = set()
    for _, h in top_b.iterrows():
        if h["doc_id"] not in seen:
            seen.add(h["doc_id"])
            src_b.append(h["doc_id"])
    src_a = []
    seen = set()
    for _, h in top_a.iterrows():
        if h["doc_id"] not in seen:
            seen.add(h["doc_id"])
            src_a.append(h["doc_id"])
    changed = "|".join(src_b) != "|".join(src_a)
    before_after_rows.append(
        {
            "query": q,
            "before_retrieved_sources": "|".join(src_b),
            "after_retrieved_sources": "|".join(src_a),
            "changed": changed,
        }
    )

ba_df = pd.DataFrame(before_after_rows)
display(ba_df)
ba_path = ARTIFACTS / "retrieval_before_after_update.csv"
ba_df.to_csv(ba_path, index=False, encoding="utf-8")
print("Сохранено:", ba_path)

chunks_df = chunks_df_after
chunk_vectors = vec_a
faiss_index = idx_a


,query,before_retrieved_sources,after_retrieved_sources,changed
0,Как rolling update настроить в Kubernetes?,precommit|github_actions|reproducibility|ci_cd...,helm|precommit|k8s_basics|github_actions|repro...,True
1,Чем Helm отличается от простого kubectl apply?,docker_intro|pytest|makefile|docker_compose|re...,helm|prometheus|docker_intro|k8s_basics|pytest,True
2,Где взять метрики латентности для алертов?,reproducibility|pytest|logging|git_basics|dock...,prometheus|reproducibility|k8s_basics|pytest|l...,True
3,Как в Docker Compose поднять стек локально?,docker_intro|docker_compose|ci_cd|reproducibil...,docker_intro|docker_compose|k8s_basics|helm|pr...,True
4,Зачем нужен pytest-cov в CI?,pytest|ci_cd|conda|git_basics|makefile,pytest|ci_cd|k8s_basics|conda|git_basics,True


Сохранено: C:\Users\Гошанский\PycharmProjects\mirea-aie\homeworks\HW14\artifacts\retrieval_before_after_update.csv


## Mini-RAG

Запрос → top-k чанков → **экстрактивный** ответ из первого чанка (первые предложения) + пометка о смежных источниках. Генеративная LLM не используется. Ответ возвращается вместе со строкой **источников** (`doc_id`).


In [9]:
def extractive_answer(query: str, hits: pd.DataFrame, max_sentences: int = 3) -> str:
    top = hits.iloc[0]
    text = top["text"]
    sentences = re.split(r"(?<=[.!?])\s+", text)
    sentences = [s.strip() for s in sentences if s.strip()]
    take = sentences[:max_sentences]
    extra = []
    if len(hits) > 1:
        extra.append("Дополнительно по смежным фрагментам: " + "; ".join(hits["doc_id"].unique()[:3]))
    return " ".join(take + extra)


def mini_rag(
    question: str,
    chunk_table: pd.DataFrame,
    index: faiss.Index,
    k: int = 4,
) -> Dict[str, object]:
    qv = encode_texts([question])
    hits = search_topk(qv, index, chunk_table, k=k)
    sources = list(dict.fromkeys(hits["doc_id"].tolist()))
    answer = extractive_answer(question, hits)
    return {"answer": answer, "retrieved_sources": "|".join(sources), "hits": hits}


RAG_QUESTIONS = [
    "Как на Windows активировать venv и зачем requirements.txt?",
    "Опиши шаги CI: от push до тестов.",
    "Что такое Prometheus pull-модель?",
    "Как Helm помогает со staging и production?",
    "Чем pod отличается от deployment в Kubernetes?",
    "Зачем correlation id в логах?",
]

rag_rows = []
for rq in RAG_QUESTIONS:
    out = mini_rag(rq, chunks_df, faiss_index, k=4)
    rag_rows.append(
        {
            "question": rq,
            "answer": out["answer"],
            "retrieved_sources": out["retrieved_sources"],
        }
    )
    print("Q:", rq)
    print("A:", out["answer"][:500], "...")
    print("Sources:", out["retrieved_sources"])
    print()

rag_df = pd.DataFrame(rag_rows)
rag_path = ARTIFACTS / "rag_examples.csv"
rag_df.to_csv(rag_path, index=False, encoding="utf-8")
print("Сохранено:", rag_path)


Q: Как на Windows активировать venv и зачем requirements.txt?
A: Модуль venv создаёт изолированный интерпретатор и каталог site-packages. Активация на Windows: .\venv\Scripts\activate, на Unix: source venv/bin/activate. После активации pip устанавливает пакеты только в окружение. Дополнительно по смежным фрагментам: venv; docker_compose; gpu_docker ...
Sources: venv|docker_compose|gpu_docker|conda

Q: Опиши шаги CI: от push до тестов.
A: CI автоматически собирает и тестирует код при каждом push. Типичный pipeline: checkout, установка зависимостей, линтер, unit-тесты, артефакты. CD расширяет процесс выкладкой в staging или production. Дополнительно по смежным фрагментам: ci_cd; pytest; precommit ...
Sources: ci_cd|pytest|precommit|github_actions

Q: Что такое Prometheus pull-модель?
A: Prometheus собирает метрики pull-моделью с /metrics endpoints. PromQL запрашивает временные ряды; Alertmanager маршрутизирует алерты. Для сервисов полезны histogram и summary для латентности. Дополнительн

## Краткий анализ ошибок

- **Смежные темы**: CI vs GitHub Actions — возможны перестановки в top-k без «ошибки» в бытовом смысле.
- **Границы чанков**: ответ может обрываться на середине мысли.
- **Экстрактивный ответ**: если в top-1 ошибочный чанк (ошибка retrieval), текст ответа будет нерелевантен.
- **Расширение базы**: новые документы конкурируют в индексе; для части запросов меняется порядок источников (`changed` в CSV).


## Сводка метрик (для отчёта)


In [10]:
import json

summary = {
    "n_docs_initial": len(RAW_DOCS),
    "n_docs_final": len(DOCS_AFTER),
    "n_chunks": int(len(chunks_df)),
    "metrics_main": {k: float(v) if isinstance(v, (float, np.floating)) else v for k, v in metrics_main.items()},
    "embed_model": EMBED_MODEL_NAME,
    "k_eval": K_EVAL,
}
print(json.dumps(summary, ensure_ascii=False, indent=2))


{
  "n_docs_initial": 15,
  "n_docs_final": 18,
  "n_chunks": 18,
  "metrics_main": {
    "hit@k": 1.0,
    "recall@k": 1.0,
    "MRR@5": 0.9333333333333332
  },
  "embed_model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
  "k_eval": 5
}
